### 1. Setup

In [ ]:
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from xgboost import XGBClassifier
import os
import time

sns.set_theme(style="whitegrid")

print("Setup complete. Ready to train models.")

### 2. Load preprocessed training and validation data

In [ ]:
print("Loading data from processed directory...")

X_train, y_train = joblib.load('../data/processed/train_data.joblib')
X_val, y_val = joblib.load('../data/processed/val_data.joblib')

print(f"Training data loaded: {X_train.shape}")
print(f"Validation data loaded: {X_val.shape}")

### 3. Random Forest (Baseline)

In [ ]:
print("Starting Random Forest training...")
start_time = time.time()

rf_model = RandomForestClassifier(
    n_estimators=100, 
    max_depth=20, 
    class_weight='balanced', 
    n_jobs=-1, 
    random_state=42
)

rf_model.fit(X_train, y_train)

train_time = time.time() - start_time
print(f"Random Forest trained in {train_time:.2f} seconds.")

rf_preds = rf_model.predict(X_val)
print("\nRandom Forest - Validation Results:")
print(classification_report(y_val, rf_preds))

### 4. XGBoost (Main model)

In [ ]:
num_benign = (y_train == 0).sum()
num_attack = (y_train == 1).sum()
scale_weight = num_benign / num_attack

print(f"Calculated scale_pos_weight: {scale_weight:.2f}")

print("Starting XGBoost training...")
start_time = time.time()

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=10,
    learning_rate=0.1,
    scale_pos_weight=scale_weight,
    use_label_encoder=False,
    eval_metric='logloss',
    n_jobs=-1,
    random_state=42
)

xgb_model.fit(X_train, y_train)

train_time = time.time() - start_time
print(f"XGBoost trained in {train_time:.2f} seconds.")

xgb_preds = xgb_model.predict(X_val)
print("\nXGBoost - Validation Results:")
print(classification_report(y_val, xgb_preds))

### 5. Result comparison

In [ ]:
cm = confusion_matrix(y_val, xgb_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - XGBoost (Validation Set)')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()

print(f"XGBoost Recall (Attack Detection): {cm[1,1]/(cm[1,1]+cm[1,0]):.4f}")
print(f"XGBoost Precision (Alarm Accuracy): {cm[1,1]/(cm[1,1]+cm[0,1]):.4f}")

### 6. Saving models

In [ ]:
os.makedirs('../models', exist_ok=True)

print("Saving models to '../models/'...")

joblib.dump(rf_model, '../models/random_forest_v1.joblib')

xgb_model.save_model('../models/xgboost_v1.json')

print("All models saved successfully!")